# KV Cache Efficiency Research Notebook (Ollama + Implemented Compression Baselines)

This notebook is a modifiable framework for KV-cache efficiency research with local Ollama models.

Included baseline methods:
- `fullkv` (no compression)
- `commonkv` (deduplicate near-duplicate sentences + keep salient sentences)
- `minicache` / `minichache` (head/tail + periodic anchors)
- `thinkkv` / `thinkk` (importance-scored sentence selection)
- `palu` (clustered sentence representatives via hashed embeddings)

> These are **implemented prompt-level proxies** for KV compression behavior, so you can run today with standard Ollama. If you later integrate true runtime KV kernels, keep the same interface and swap method internals.

## 1) Setup

In [ ]:
# !pip install -U pandas numpy matplotlib requests tqdm

import math
import re
import time
import statistics
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

## 2) Robust Ollama client (fixes `/api/generate` 404 by fallback routing)

In [ ]:
class OllamaClient:
    """Ollama client with endpoint auto-fallback.

    Handles these server styles:
      1) Native Ollama API (e.g., /api/tags, /api/generate, /api/chat)
      2) OpenAI-compatible API (e.g., /v1/models, /v1/chat/completions)
    """

    def __init__(self, base_url: str = "http://localhost:11434", timeout_s: int = 300):
        self.base_url = base_url.rstrip("/")
        self.timeout_s = timeout_s

    def _get(self, path: str):
        return requests.get(f"{self.base_url}{path}", timeout=self.timeout_s)

    def _post(self, path: str, payload: Dict[str, Any]):
        return requests.post(f"{self.base_url}{path}", json=payload, timeout=self.timeout_s)

    def list_models(self) -> List[str]:
        # Native Ollama
        try:
            r = self._get("/api/tags")
            if r.ok:
                data = r.json()
                return [m.get("name") for m in data.get("models", []) if m.get("name")]
        except Exception:
            pass

        # OpenAI-compatible fallback
        try:
            r = self._get("/v1/models")
            if r.ok:
                data = r.json()
                return [m.get("id") for m in data.get("data", []) if m.get("id")]
        except Exception:
            pass

        return []

    def generate(self, model: str, prompt: str, options: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        options = options or {}

        # Attempt 1: native /api/generate
        try:
            r = self._post("/api/generate", {
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": options,
            })
            if r.ok:
                data = r.json()
                return {
                    "response": data.get("response", ""),
                    "raw": data,
                    "endpoint": "/api/generate",
                }
            if r.status_code not in (404, 405):
                r.raise_for_status()
        except requests.HTTPError:
            raise
        except Exception:
            pass

        # Attempt 2: native /api/chat
        try:
            r = self._post("/api/chat", {
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "stream": False,
                "options": options,
            })
            if r.ok:
                data = r.json()
                msg = data.get("message", {}) if isinstance(data, dict) else {}
                return {
                    "response": msg.get("content", ""),
                    "raw": data,
                    "endpoint": "/api/chat",
                }
            if r.status_code not in (404, 405):
                r.raise_for_status()
        except requests.HTTPError:
            raise
        except Exception:
            pass

        # Attempt 3: OpenAI-compatible endpoint
        r = self._post("/v1/chat/completions", {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": options.get("temperature", 0.2),
            "max_tokens": options.get("num_predict", 256),
        })
        r.raise_for_status()
        data = r.json()
        text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
        return {
            "response": text,
            "raw": data,
            "endpoint": "/v1/chat/completions",
        }


CLIENT = OllamaClient(base_url="http://localhost:11434", timeout_s=300)
LOCAL_MODELS = CLIENT.list_models()
print("LOCAL MODELS FOUND:", LOCAL_MODELS)

## 3) Text utilities for compression baselines

In [ ]:
WORD_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def tokenize(text: str) -> List[str]:
    return WORD_RE.findall(text)


def detokenize(tokens: List[str]) -> str:
    out = []
    for t in tokens:
        if out and re.match(r"[\w]", t) and re.match(r"[\w]", out[-1][-1]):
            out.append(" " + t)
        elif out and t not in ".,!?;:)%]}" and out[-1][-1] not in "([{":
            out.append(" " + t)
        else:
            out.append(t)
    return "".join(out)


def split_sentences(text: str) -> List[str]:
    s = re.split(r"(?<=[.!?\n])\s+", text.strip())
    return [x.strip() for x in s if x.strip()]


def normalize_for_set(sentence: str) -> set:
    toks = [t.lower() for t in tokenize(sentence) if re.match(r"\w+", t)]
    return set(toks)


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0


def estimate_tokens(text: str) -> int:
    return max(1, math.ceil(len(text) / 4)) if text else 0

## 4) Method interface + implemented baselines

In [ ]:
@dataclass
class GenerationResult:
    model: str
    method: str
    prompt_id: str
    original_chars: int
    compressed_chars: int
    compression_ratio: float
    prompt_tokens_est: int
    output_text: str
    output_tokens_est: int
    latency_s: float
    endpoint_used: str
    options: Dict[str, Any] = field(default_factory=dict)
    meta: Dict[str, Any] = field(default_factory=dict)


class KVCompressionMethod:
    name = "base"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        return prompt

    def ollama_options(self, config: Dict[str, Any]) -> Dict[str, Any]:
        return {}


class FullKV(KVCompressionMethod):
    name = "fullkv"


class CommonKV(KVCompressionMethod):
    """CommonKV proxy: remove near-duplicate sentences, keep salient remainder."""
    name = "commonkv"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        sentences = split_sentences(prompt)
        if len(sentences) <= 3:
            return prompt

        sim_th = config.get("commonkv_similarity_threshold", 0.88)
        keep_budget = config.get("commonkv_keep_sentences", max(8, len(sentences) // 2))

        kept = []
        kept_sets = []
        for s in sentences:
            sset = normalize_for_set(s)
            if any(jaccard(sset, x) >= sim_th for x in kept_sets):
                continue
            kept.append(s)
            kept_sets.append(sset)

        # Salience score: unique-token ratio + digit/code hints.
        scored = []
        for s in kept:
            toks = [t for t in tokenize(s) if re.match(r"\w+", t)]
            if not toks:
                continue
            uniq_ratio = len(set(t.lower() for t in toks)) / len(toks)
            has_digits = 1.0 if re.search(r"\d", s) else 0.0
            has_code = 1.0 if re.search(r"[{}();=<>]", s) else 0.0
            score = uniq_ratio + 0.2 * has_digits + 0.2 * has_code
            scored.append((score, s))

        scored.sort(key=lambda x: x[0], reverse=True)
        selected = [s for _, s in scored[:keep_budget]]

        # preserve original order
        order = {s: i for i, s in enumerate(sentences)}
        selected.sort(key=lambda s: order.get(s, 10**9))
        return "
".join(selected)


class MiniCache(KVCompressionMethod):
    """MiniCache proxy: head/tail retention + periodic anchor spans."""
    name = "minicache"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        toks = tokenize(prompt)
        n = len(toks)
        if n == 0:
            return prompt

        head = config.get("minicache_head_tokens", 700)
        tail = config.get("minicache_tail_tokens", 500)
        stride = config.get("minicache_anchor_stride", 400)
        anchor = config.get("minicache_anchor_width", 60)

        keep_idx = set(range(min(head, n)))
        keep_idx.update(range(max(0, n - tail), n))

        i = head
        while i < max(head, n - tail):
            keep_idx.update(range(i, min(i + anchor, n)))
            i += max(anchor, stride)

        compressed = [toks[i] for i in sorted(keep_idx)]
        return detokenize(compressed)


class ThinkKV(KVCompressionMethod):
    """ThinkKV proxy: importance-ranked sentence selection with edge preservation."""
    name = "thinkkv"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        sentences = split_sentences(prompt)
        if len(sentences) <= 4:
            return prompt

        head_keep = config.get("thinkkv_head_sentences", 2)
        tail_keep = config.get("thinkkv_tail_sentences", 2)
        budget = config.get("thinkkv_total_sentences", max(8, len(sentences) // 3))

        middle = sentences[head_keep: max(head_keep, len(sentences) - tail_keep)]

        scored = []
        for s in middle:
            toks = [t.lower() for t in tokenize(s) if re.match(r"\w+", t)]
            if not toks:
                continue
            uniq = len(set(toks)) / len(toks)
            length_bonus = min(len(toks) / 40.0, 1.0)
            entity_bonus = 0.3 if re.search(r"[A-Z][a-z]+", s) else 0.0
            num_bonus = 0.3 if re.search(r"\d", s) else 0.0
            score = uniq + 0.4 * length_bonus + entity_bonus + num_bonus
            scored.append((score, s))

        scored.sort(key=lambda x: x[0], reverse=True)
        need_middle = max(0, budget - head_keep - tail_keep)
        selected_middle = [s for _, s in scored[:need_middle]]

        order = {s: i for i, s in enumerate(sentences)}
        selected_middle.sort(key=lambda s: order[s])

        result = sentences[:head_keep] + selected_middle + sentences[-tail_keep:]
        return "
".join(result)

    def ollama_options(self, config: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "temperature": config.get("thinkkv_temperature", 0.1),
            "num_ctx": config.get("thinkkv_num_ctx", config.get("num_ctx", 8192)),
        }


def _hash_embed(text: str, dim: int = 128) -> np.ndarray:
    vec = np.zeros(dim, dtype=float)
    for tok in tokenize(text):
        if not re.match(r"\w+", tok):
            continue
        h = hash(tok.lower()) % dim
        vec[h] += 1.0
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec /= norm
    return vec


class PaLu(KVCompressionMethod):
    """PaLu proxy: sentence clustering and representative selection."""
    name = "palu"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        sentences = split_sentences(prompt)
        if len(sentences) <= 6:
            return prompt

        k = int(config.get("palu_clusters", min(12, max(4, len(sentences) // 12))))
        emb_dim = int(config.get("palu_embed_dim", 128))
        iters = int(config.get("palu_kmeans_iters", 6))

        X = np.vstack([_hash_embed(s, dim=emb_dim) for s in sentences])
        rng = np.random.default_rng(config.get("seed", 42))
        init_idx = rng.choice(len(sentences), size=min(k, len(sentences)), replace=False)
        C = X[init_idx].copy()

        for _ in range(iters):
            sims = X @ C.T
            labels = sims.argmax(axis=1)
            for ci in range(C.shape[0]):
                members = X[labels == ci]
                if len(members) > 0:
                    C[ci] = members.mean(axis=0)
                    n = np.linalg.norm(C[ci])
                    if n > 0:
                        C[ci] /= n

        sims = X @ C.T
        labels = sims.argmax(axis=1)

        selected = []
        for ci in range(C.shape[0]):
            idx = np.where(labels == ci)[0]
            if len(idx) == 0:
                continue
            best_local = idx[(X[idx] @ C[ci]).argmax()]
            selected.append(best_local)

        # also retain first and last to preserve context edges
        selected.extend([0, len(sentences) - 1])
        selected = sorted(set(selected))

        return "
".join(sentences[i] for i in selected)


METHODS = {
    "fullkv": FullKV(),
    "commonkv": CommonKV(),
    "minicache": MiniCache(),
    "minichache": MiniCache(),
    "thinkkv": ThinkKV(),
    "thinkk": ThinkKV(),
    "palu": PaLu(),
}

print("Methods:", sorted(METHODS))

## 5) Runner

In [ ]:
def run_one(model: str, method_name: str, prompt_id: str, prompt: str, config: Dict[str, Any]) -> GenerationResult:
    method = METHODS[method_name]

    compressed = method.preprocess_prompt(prompt, config)
    options = method.ollama_options(config)

    t0 = time.perf_counter()
    resp = CLIENT.generate(model=model, prompt=compressed, options=options)
    latency = time.perf_counter() - t0

    output_text = resp.get("response", "")

    return GenerationResult(
        model=model,
        method=method_name,
        prompt_id=prompt_id,
        original_chars=len(prompt),
        compressed_chars=len(compressed),
        compression_ratio=(len(compressed) / len(prompt)) if len(prompt) else 1.0,
        prompt_tokens_est=estimate_tokens(compressed),
        output_text=output_text,
        output_tokens_est=estimate_tokens(output_text),
        latency_s=latency,
        endpoint_used=resp.get("endpoint", "unknown"),
        options=options,
        meta={"raw": resp.get("raw", {})},
    )


def run_grid(prompts: Dict[str, str], models: List[str], methods: List[str], config: Optional[Dict[str, Any]] = None) -> pd.DataFrame:
    config = config or {}
    rows = []

    total = len(prompts) * len(models) * len(methods)
    with tqdm(total=total, desc="KV experiments") as pbar:
        for model in models:
            for pid, prompt in prompts.items():
                for method in methods:
                    try:
                        r = run_one(model=model, method_name=method, prompt_id=pid, prompt=prompt, config=config)
                        rows.append({
                            "model": r.model,
                            "method": r.method,
                            "prompt_id": r.prompt_id,
                            "original_chars": r.original_chars,
                            "compressed_chars": r.compressed_chars,
                            "compression_ratio": r.compression_ratio,
                            "prompt_tokens_est": r.prompt_tokens_est,
                            "output_tokens_est": r.output_tokens_est,
                            "latency_s": r.latency_s,
                            "tok_per_s_est": (r.output_tokens_est / r.latency_s) if r.latency_s else np.nan,
                            "endpoint_used": r.endpoint_used,
                            "output_preview": r.output_text[:200],
                        })
                    except Exception as e:
                        rows.append({
                            "model": model,
                            "method": method,
                            "prompt_id": pid,
                            "original_chars": len(prompt),
                            "compressed_chars": np.nan,
                            "compression_ratio": np.nan,
                            "prompt_tokens_est": np.nan,
                            "output_tokens_est": np.nan,
                            "latency_s": np.nan,
                            "tok_per_s_est": np.nan,
                            "endpoint_used": "error",
                            "output_preview": f"ERROR: {e}",
                        })
                    pbar.update(1)

    return pd.DataFrame(rows)

## 6) Configure experiments

In [ ]:
SELECTED_MODELS = LOCAL_MODELS if LOCAL_MODELS else ["qwen3:8b"]

PROMPTS = {
    "long_context_analysis": (
        "Summarize key technical points and produce 5 follow-up research ideas.\n\n"
        + "Transformer inference bottlenecks often involve memory traffic and KV growth. " * 500
    ),
    "code_style_reasoning": (
        "Analyze this pseudo-code and suggest optimization strategy with complexity notes.\n\n"
        + "for layer in layers: for token in seq: update_kv(layer, token)\n" * 220
    ),
}

METHOD_LIST = ["fullkv", "commonkv", "minicache", "thinkkv", "palu"]

CONFIG = {
    "num_ctx": 8192,
    "seed": 42,

    # commonkv
    "commonkv_similarity_threshold": 0.88,
    "commonkv_keep_sentences": 140,

    # minicache
    "minicache_head_tokens": 700,
    "minicache_tail_tokens": 500,
    "minicache_anchor_stride": 400,
    "minicache_anchor_width": 60,

    # thinkkv
    "thinkkv_head_sentences": 2,
    "thinkkv_tail_sentences": 2,
    "thinkkv_total_sentences": 120,
    "thinkkv_temperature": 0.1,
    "thinkkv_num_ctx": 4096,

    # palu
    "palu_clusters": 10,
    "palu_embed_dim": 128,
    "palu_kmeans_iters": 6,
}

results_df = run_grid(
    prompts=PROMPTS,
    models=SELECTED_MODELS,
    methods=METHOD_LIST,
    config=CONFIG,
)

results_df.head(20)

## 7) Summary metrics

In [ ]:
summary = (
    results_df
    .groupby(["model", "method"], as_index=False)
    .agg(
        runs=("latency_s", "count"),
        mean_latency_s=("latency_s", "mean"),
        median_latency_s=("latency_s", "median"),
        mean_compression_ratio=("compression_ratio", "mean"),
        mean_output_tokens=("output_tokens_est", "mean"),
        mean_tps=("tok_per_s_est", "mean"),
    )
    .sort_values(["model", "mean_latency_s"])
)
summary

## 8) Plots

In [ ]:
import matplotlib.pyplot as plt

for model in summary["model"].unique():
    sub = summary[summary["model"] == model]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].bar(sub["method"], sub["mean_latency_s"])
    axes[0].set_title(f"{model} latency")
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].bar(sub["method"], sub["mean_tps"])
    axes[1].set_title(f"{model} throughput")
    axes[1].tick_params(axis="x", rotation=45)

    axes[2].bar(sub["method"], sub["mean_compression_ratio"])
    axes[2].set_title(f"{model} compression ratio")
    axes[2].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

## 9) Save results

In [ ]:
out_dir = Path("artifacts")
out_dir.mkdir(exist_ok=True)

results_path = out_dir / "kv_experiment_results.csv"
summary_path = out_dir / "kv_experiment_summary.csv"

results_df.to_csv(results_path, index=False)
summary.to_csv(summary_path, index=False)

print("Saved:")
print(" -", results_path)
print(" -", summary_path)

## 10) Add your own KV idea

1. Create a new class inheriting `KVCompressionMethod`.
2. Implement `preprocess_prompt` (or replace with true backend KV integration).
3. Optionally override `ollama_options` for context/decoding controls.
4. Register it in `METHODS` and add to `METHOD_LIST`.

Because all methods run through the same `run_grid` and metrics pipeline, your comparisons remain consistent.